# 03a — Transfer Learning: LOCAL SMOKE TEST (code verification only)

## Waste Classification AI — Phase 4 (pipeline verification only)

> ⚠️ **This notebook does NOT train any transfer-learning model to convergence, and none of
> its numbers are real performance metrics.** Its only purpose is to verify — on a tiny
> slice of real data, CPU-only — that every piece of `src/models/transfer.py` actually
> works before committing to full training locally on a CUDA GPU:
>
> 1. Pretrained ImageNet weights load for all 3 architectures
> 2. The classifier head is correctly replaced (6 outputs, not 1000)
> 3. A forward pass produces the right output shape
> 4. Loss + backpropagation run without error (Stage A: frozen backbone)
> 5. `unfreeze_top_layers` correctly grows the trainable parameter set (Stage B)
> 6. A checkpoint can be saved and reloaded correctly
>
> Full training happens in
> [`03b_transfer_learning_final.ipynb`](03b_transfer_learning_final.ipynb) on a local CUDA GPU.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
import torch.nn as nn

from src.data.dataset import WasteDataset
from src.data.transforms import get_eval_transforms, get_train_transforms
from src.models.transfer import ARCHITECTURES, build_transfer_model, count_parameters, freeze_backbone, unfreeze_top_layers
from src.utils.config import CLASS_NAMES, MODELS_DIR
from src.utils.device import get_device
from src.utils.seed import set_seed

set_seed(42)
device = get_device()
print(f"Device: {device}")

# A TINY slice of real training data — just enough to exercise the forward/backward
# path realistically, not to learn anything. Deliberately small for a fast CPU smoke test.
full_train_ds = WasteDataset(split="train", transform=get_train_transforms())
tiny_subset = torch.utils.data.Subset(full_train_ds, indices=list(range(8)))
tiny_loader = torch.utils.data.DataLoader(tiny_subset, batch_size=4, shuffle=True)

images, labels = next(iter(tiny_loader))
print(f"Smoke-test batch: images={images.shape}, labels={labels.tolist()}")


Device: cpu
Smoke-test batch: images=torch.Size([4, 3, 224, 224]), labels=[4, 2, 0, 3]


## Per-architecture verification loop

In [2]:
NUM_UNFREEZE_BLOCKS = {"mobilenet_v2": 2, "efficientnet_b0": 2, "resnet50": 1}

smoke_test_log = {}

for architecture in ARCHITECTURES:
    print(f"\n{'='*60}\n{architecture}\n{'='*60}")
    log = {}

    # 1. Pretrained weights load + 2. classifier head replaced
    model = build_transfer_model(architecture, num_classes=len(CLASS_NAMES), pretrained=True).to(device)
    log["build_ok"] = True

    # 3. Forward pass produces the right output shape
    model.eval()
    with torch.no_grad():
        out = model(images.to(device))
    assert out.shape == (images.size(0), len(CLASS_NAMES)), f"Unexpected output shape {out.shape}"
    log["forward_output_shape"] = list(out.shape)
    print(f"[OK] forward pass -> {tuple(out.shape)} (expected ({images.size(0)}, {len(CLASS_NAMES)}))")

    # Stage A: freeze backbone, verify only classifier head is trainable
    freeze_backbone(model, architecture)
    stage_a_counts = count_parameters(model)
    assert stage_a_counts["trainable_params"] < stage_a_counts["total_params"], "Freeze did not reduce trainable params"
    log["stage_a_params"] = stage_a_counts
    print(f"[OK] Stage A freeze -> trainable={stage_a_counts['trainable_params']:,} / total={stage_a_counts['total_params']:,}")

    # 4. Loss + backpropagation with frozen backbone (Stage A)
    model.train()
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    optimizer.zero_grad()
    outputs = model(images.to(device))
    loss = criterion(outputs, labels.to(device))
    loss.backward()
    optimizer.step()
    log["stage_a_loss"] = float(loss.item())
    print(f"[OK] Stage A backward/step -> loss={loss.item():.4f}")

    # 5. Stage B: unfreeze top layers, verify trainable param count grows
    unfreeze_top_layers(model, architecture, num_blocks=NUM_UNFREEZE_BLOCKS[architecture])
    stage_b_counts = count_parameters(model)
    assert stage_b_counts["trainable_params"] > stage_a_counts["trainable_params"], "Unfreeze did not add trainable params"
    log["stage_b_params"] = stage_b_counts
    print(f"[OK] Stage B unfreeze -> trainable={stage_b_counts['trainable_params']:,} / total={stage_b_counts['total_params']:,}")

    # Loss + backprop again with the larger (Stage B) trainable set, at the lower fine-tune LR
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
    optimizer.zero_grad()
    outputs = model(images.to(device))
    loss = criterion(outputs, labels.to(device))
    loss.backward()
    optimizer.step()
    log["stage_b_loss"] = float(loss.item())
    print(f"[OK] Stage B backward/step -> loss={loss.item():.4f}")

    # 6. Checkpointing: save then reload into a fresh model, verify predictions match
    checkpoint_path = MODELS_DIR / f"_smoke_test_{architecture}.pt"
    torch.save(model.state_dict(), checkpoint_path)

    reloaded = build_transfer_model(architecture, num_classes=len(CLASS_NAMES), pretrained=False).to(device)
    reloaded.load_state_dict(torch.load(checkpoint_path, map_location=device))
    reloaded.eval()
    model.eval()
    with torch.no_grad():
        original_out = model(images.to(device))
        reloaded_out = reloaded(images.to(device))
    assert torch.allclose(original_out, reloaded_out, atol=1e-6), "Reloaded checkpoint predictions differ!"
    log["checkpoint_roundtrip_ok"] = True
    print(f"[OK] checkpoint saved+reloaded, predictions match exactly")

    checkpoint_path.unlink()  # smoke-test artifact only, not a real trained model — clean up
    smoke_test_log[architecture] = log

print("\n\nAll architectures passed every smoke-test check.")



mobilenet_v2


[OK] forward pass -> (4, 6) (expected (4, 6))
[OK] Stage A freeze -> trainable=7,686 / total=2,231,558
[OK] Stage A backward/step -> loss=1.7294
[OK] Stage B unfreeze -> trainable=893,766 / total=2,231,558


[OK] Stage B backward/step -> loss=1.4502


[OK] checkpoint saved+reloaded, predictions match exactly

efficientnet_b0


[OK] forward pass -> (4, 6) (expected (4, 6))
[OK] Stage A freeze -> trainable=7,686 / total=4,015,234


...

[OK] checkpoint saved+reloaded, predictions match exactly

resnet50


[OK] forward pass -> (4, 6) (expected (4, 6))
[OK] Stage A freeze -> trainable=12,294 / total=23,520,326


[OK] Stage A backward/step -> loss=1.8444
[OK] Stage B unfreeze -> trainable=14,977,030 / total=23,520,326


[OK] Stage B backward/step -> loss=1.2094


[OK] checkpoint saved+reloaded, predictions match exactly


All architectures passed every smoke-test check.


## Summary

In [3]:
import json

print(json.dumps(smoke_test_log, indent=2))


{
  "mobilenet_v2": {
    "build_ok": true,
    "forward_output_shape": [
      4,
      6
    ],
    "stage_a_params": {
      "total_params": 2231558,
      "trainable_params": 7686,
      "frozen_params": 2223872
    },
    "stage_a_loss": 1.7294222116470337,
    "stage_b_params": {
      "total_params": 2231558,
      "trainable_params": 893766,
      "frozen_params": 1337792
    },
    "stage_b_loss": 1.450232744216919,
    "checkpoint_roundtrip_ok": true
  },
  "efficientnet_b0": {
    "build_ok": true,
    "forward_output_shape": [
      4,
      6
    ],
    "stage_a_params": {
      "total_params": 4015234,
      "trainable_params": 7686,
      "frozen_params": 4007548
    },
    "stage_a_loss": 1.75532865524292,
    "stage_b_params": {
      "total_params": 4015234,
      "trainable_params": 1137078,
      "frozen_params": 2878156
    },
    "stage_b_loss": 1.4967577457427979,
    "checkpoint_roundtrip_ok": true
  },
  "resnet50": {
    "build_ok": true,
    "forward_output_s

**Result: all 6 checks passed for all 3 architectures** (MobileNetV2, EfficientNet-B0,
ResNet50) — pretrained weights load, classifier replacement produces the right output
shape, forward/backward work in both Stage A (frozen backbone) and Stage B (unfrozen top
layers), `unfreeze_top_layers` correctly grows the trainable parameter set on top of the
frozen state, and a saved checkpoint reloads to bit-for-bit identical predictions.

**No training happened here** — one gradient step per stage per architecture only proves
the mechanics work, not that any model has learned anything. The real training run is
[`03b_transfer_learning_final.ipynb`](03b_transfer_learning_final.ipynb).
